# Dynamic Gold Dimension Maker

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

## Parameters

### airports

In [0]:
# ================ SOURCE ================ 
catalog_source = "flyproject"
schema_source = "silver"
table_source = "silver_airports"

# ================ TARGET ================ 
catalog_target = "flyproject"
schema_target = "gold"
table_target = "dim_airports"

# ================ COLUMN/ADDITIONAL ================ 
key_column = ["airport_id"]
surrogate_key = "DimAirportsKey"
cdc_column = "processed_timestamp"
backdated_refresh = ""

### flights

In [0]:
# # ================ SOURCE ================ 
# catalog_source = "flyproject"
# schema_source = "silver"
# table_source = "silver_flights"

# # ================ TARGET ================ 
# catalog_target = "flyproject"
# schema_target = "gold"
# table_target = "dim_flights"

# # ================ COLUMN/ADDITIONAL ================ 
# key_column = ["flight_id"]
# surrogate_key = "DimFlightsKey"
# cdc_column = "processed_timestamp"
# backdated_refresh = ""

### passengers

In [0]:
# # ================ SOURCE ================ 
# catalog_source = "flyproject"
# schema_source = "silver"
# table_source = "silver_passengers"

# # ================ TARGET ================ 
# catalog_target = "flyproject"
# schema_target = "gold"
# table_target = "dim_passengers"

# # ================ COLUMN/ADDITIONAL ================ 
# key_column = ["passenger_id"]
# surrogate_key = "DimPassengersKey"
# cdc_column = "processed_timestamp"
# backdated_refresh = ""

In [0]:
# --------- Eksperimen --------- 

data01 = ["aaa", "bbb", "ccc"]
print("-".join(data01))

data02 = ["airport_id", "airport_code"]
", ".join(data02)

# jadi join --> mengunpack list menjadi string

## Mencari Last Load Untuk DF Source

In [0]:
if len(backdated_refresh) == 0:

    if spark.catalog.tableExists(f"{catalog_target}.{schema_target}.{table_target}"):
        last_load = spark.sql(f"""
            SELECT max({cdc_column})
            FROM {catalog_target}.{schema_target}.{table_target}
        """).collect()[0][0]

    else:
        last_load = "1900-01-01 00:00:00"

else:
    last_load = backdated_refresh

In [0]:
# Menangkap last_load untuk debugging
last_load

In [0]:
# DF Source --> Df yang akan kita proses
df_source = spark.sql(f"""
                      SELECT * 
                      FROM {catalog_source}.{schema_source}.{table_source} 
                      WHERE {cdc_column} > '{last_load}'
                      """)

In [0]:
display(df_source)

## Menyiapkan DF Target Yang Nantinya Akan Di JOIN dengan DF Source
Supaya tahu mana records yang sudah ada di target dan mana record yang baru. Simpelnya untuk tahu mana yang di Update (Old) dan mana yang di Insert (New)

In [0]:
# Ada 2 Skenario: Incremental Load & Initial Load
# Check apakah Gold Dimension sudah ada
# Jika ada -> Incremental Load (Ambil df target dari gold)
# Jika belum -> Initial Load (Buat df target baru untuk placeholder)

if spark.catalog.tableExists(f"{catalog_target}.{schema_target}.{table_target}"):

    # Incremental Load
    df_target = spark.sql(f"""
                          SELECT
                            {", ".join(key_column)},
                            {surrogate_key},
                            create_date,
                            update_date
                          FROM {catalog_target}.{schema_target}.{table_target}
                          """)
    
else:

    # Initial Load
    schema = []

    for key in key_column:
        schema.append(StructField(key, StringType()))

    schema.extend([
        StructField(surrogate_key, LongType()),
        StructField("create_date", TimestampType()),
        StructField("update_date", TimestampType())
    ])

    df_target = spark.createDataFrame([], StructType(schema))

In [0]:
# Debugging
print(schema)
display(df_target)

## Join DF Source dengan DF Target

In [0]:
# Pastikan schema JOIN KEY nya sama supaya bisa di JOIN
df_target.printSchema()
df_source.printSchema()

In [0]:
# Kita jadikan temporary view supaya kita bisa JOIN menggunakan SQL
df_source.createOrReplaceTempView("df_source")
df_target.createOrReplaceTempView("df_target")

# Kita buatkan Join Condition misal key_column nya lebih dari 1
join_condition = " AND ".join([f"df_source.{key} = df_target.{key}" for key in key_column])
join_condition

In [0]:
df_join = spark.sql(f"""
                  SELECT
                    df_source.*,
                    df_target.{surrogate_key},
                    df_target.create_date,
                    df_target.update_date
                  FROM df_source
                  LEFT JOIN df_target
                  ON {join_condition}
                  """)

display(df_join)

## Memisahkan Antara Existing Record & New Record
Agar tau mana yang di Update (old) dan mana yang di Insert (New). Mereka mempunyai perlakuan khusus masing2  
- Old Record  --> Update the update_date  
- New Record --> Menetapkan DimSurrogateKey, Mengisi create_date & update_date (karena belum ada)

### Old (Existing) Records

In [0]:
df_existing_record = df_join.filter(df_join[surrogate_key].isNotNull())
display(df_existing_record)

In [0]:
df_existing_record = df_existing_record.withColumn("update_date", lit(current_timestamp()))
display(df_existing_record)

### New Records

In [0]:
# INGAT!! di New Record kita Mengenrich Column yang masih kosong:
# Menetapkan DimSurrogateKey, Mengisi create_date & update_date (karena belum ada)

df_new_record = df_join.filter(col(f'{surrogate_key}').isNull())
display(df_new_record)

In [0]:
# Di New Record ada 2 kemungkinan:
# 1. New Record dari Incremental Load
# 2. New Record karena Initial Load


if spark.catalog.tableExists(f"{catalog_target}.{schema_target}.{table_target}"):

    # Incremental Load
    max_surrogate_key = spark.sql(f"""
                                  SELECT MAX({surrogate_key})
                                  FROM {catalog_target}.{schema_target}.{table_target}
                                  """).collect()[0][0]
    df_new_record = (
        df_new_record
        .withColumn(f"{surrogate_key}", lit(max_surrogate_key) + (monotonically_increasing_id() + lit(1)))
        .withColumn("create_date", current_timestamp())
        .withColumn("update_date", current_timestamp())
    )
    

else:
    # Initial Load
    df_new_record = (
        df_new_record
        .withColumn(f"{surrogate_key}", monotonically_increasing_id() + lit(1))
        .withColumn("create_date", current_timestamp())
        .withColumn("update_date", current_timestamp())
    )


In [0]:
display(df_new_record)

## Unioning Processed Existing Record & New Record
Meng-Union kan df_existing_record & df_new_record karena mereka sudah diolah dan siap digabung  
menjadi df_union/df_final yang siap di UPSERT ke DF Target (Gold Dimension)

In [0]:
# Kalau Union KEDUA DF Schema nya harus SAMA PERSIS
df_existing_record.printSchema()
df_new_record.printSchema()

In [0]:
df_final = df_existing_record.unionByName(df_new_record)

# Menaruh Surrogate Key di depan
metadata_columns = ["create_date", "update_date"]

other_columns = [
    column
    for column in df_final.columns
    if column not in key_column + [surrogate_key] + metadata_columns
]

df_final = df_final.select(
    surrogate_key,
    *key_column,
    *other_columns,
    *metadata_columns
)

# metadata_columns  --> ["create_date", "update_date"]
# *metadata_columns --> "create_date", "update_date"
# Jadi * membongkar isi list menjadi argumen satu per satu.

display(df_final)

## Loading Final DF into Gold Dimensions

In [0]:
# Ada 2 Skenario:
# Jika tabel Gold sudah ada -> lakukan UPSERT
# Jika belum ada -> buat tabel baru (Initial Load)


if spark.catalog.tableExists(f"{catalog_target}.{schema_target}.{table_target}"):

    # Update record lama dan insert record baru
    target_table = DeltaTable.forName(spark, f"{catalog_target}.{schema_target}.{table_target}")
    target_table.alias("target") \
                .merge(df_final.alias("source"), f"target.{surrogate_key} = source.{surrogate_key}") \
                .whenMatchedUpdateAll(condition = f"source.{cdc_column} >= target.{cdc_column}") \
                .whenNotMatchedInsertAll() \
                .execute()


else:
    # Initial Load
    df_final.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(f"{catalog_target}.{schema_target}.{table_target}")


# Kalau surrogate key sama
# ↓
# anggap record yang sama

# Kalau source lebih baru
# ↓
# UPDATE

In [0]:
%sql
SELECT * FROM flyproject.gold.dim_airports WHERE airport_id = "A023"